# Production Trade Report

Pull historical trade data directly from the local MetaTrader 5 terminal and summarise production performance by symbol.

**Workflow**
1. Set the date range and optional symbol/group filters
2. Run the fetch cell to load MT5 deal history
3. Review the per-symbol summary, daily PnL, and best/worst trades


In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from Learn.report import fetch_trade_report

In [2]:
# -- Configuration -----------------------------------------------------------
SYMBOL      = "XAUUSD"  # Base symbol name (must match data/{SYMBOL}_M1_520weeks.csv)

START_DATE  = "2026-04-19"
END_DATE    = pd.Timestamp.now("UTC").strftime("%Y-%m-%d")

GROUP       = "*"      # MT5 group filter, e.g. "*USD*" or "*"
SYMBOLS     = None     # e.g. ["EURUSD.a", "US500.a"]
CLOSED_ONLY = True     # Summary focuses on close-side deals only


In [3]:
trades, summary = fetch_trade_report(
    start_date=START_DATE,
    end_date=END_DATE,
    group=GROUP,
    symbols=SYMBOLS,
    closed_only=CLOSED_ONLY,
)

closed_trades = trades[trades["entry_type"].isin(["OUT", "OUT_BY", "INOUT"])].copy()
closed_trades["trade_date"] = closed_trades["time"].dt.tz_convert(None).dt.floor("D") if not closed_trades.empty else pd.Series(dtype="datetime64[ns]")


In [4]:
print(f"Rows returned      : {len(trades):,}")
print(f"Closed-trade rows  : {len(closed_trades):,}")
print(f"Symbols returned   : {sorted(trades['symbol'].dropna().unique().tolist()) if not trades.empty else []}")
if not trades.empty:
    print(f"Time range (UTC)   : {trades['time'].min()} -> {trades['time'].max()}")

display(trades.head(10))

Rows returned      : 184
Closed-trade rows  : 92
Symbols returned   : ['EURUSD.a', 'US500.a', 'XAUUSD.a']
Time range (UTC)   : 2026-04-20 04:59:14+00:00 -> 2026-04-24 21:22:56+00:00


,ticket,order,position_id,time,time_msc,symbol,side,deal_type,entry_type,reason,volume,price,profit,commission,swap,fee,net_pnl,magic,comment,external_id
0,230530595,289625914,289625914,2026-04-20 04:59:14+00:00,2026-04-20 04:59:14.459000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17576,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
1,230539380,289639884,289625914,2026-04-20 05:30:01+00:00,2026-04-20 05:30:01.090000+00:00,EURUSD.a,sell,SELL,OUT,TP,1.25,1.17606,52.41,-4.38,0.0,0.0,48.03,235000,[tp 1.17606],
2,230760881,289947428,289947428,2026-04-20 15:23:12+00:00,2026-04-20 15:23:12.598000+00:00,EURUSD.a,sell,SELL,IN,EXPERT,1.25,1.17602,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
3,230772630,289961383,289947428,2026-04-20 15:41:57+00:00,2026-04-20 15:41:57.570000+00:00,EURUSD.a,buy,BUY,OUT,SL,1.25,1.17646,-76.84,-4.38,0.0,0.0,-81.22,235000,[sl 1.17646],
4,230938194,290155348,290155348,2026-04-20 18:54:12+00:00,2026-04-20 18:54:12.900000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17835,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
5,230950043,290170311,290155348,2026-04-20 19:19:07+00:00,2026-04-20 19:19:07.585000+00:00,EURUSD.a,sell,SELL,OUT,TP,1.25,1.17881,80.14,-4.38,0.0,0.0,75.76,235000,[tp 1.17881],
6,230990855,290227844,290227844,2026-04-20 21:28:00+00:00,2026-04-20 21:28:00.855000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17873,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
7,230991010,290228358,290227844,2026-04-20 21:28:39+00:00,2026-04-20 21:28:39.383000+00:00,EURUSD.a,sell,SELL,OUT,SL,1.25,1.17841,-55.74,-4.38,0.0,0.0,-60.12,235000,[sl 1.17841],
8,231118541,290422193,290422193,2026-04-21 06:54:09+00:00,2026-04-21 06:54:09.403000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4796.45000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,
9,231120367,290424929,290424929,2026-04-21 07:02:24+00:00,2026-04-21 07:02:24.425000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4797.02000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,


## Per-symbol summary

In [5]:
if summary.empty:
    print("No symbol summary available for the selected range/filter.")
else:
    summary_display = summary.copy()
    summary_display["win_rate"] = (summary_display["win_rate"] * 100).round(2)
    display(summary_display.round({
        "volume_lots": 2,
        "gross_profit": 2,
        "gross_loss": 2,
        "net_pnl": 2,
        "avg_net_pnl": 2,
        "median_net_pnl": 2,
        "win_rate": 2,
        "avg_win": 2,
        "avg_loss": 2,
        "total_commission": 2,
        "total_swap": 2,
        "total_fee": 2,
    }))

,symbol,trade_count,first_trade_time,last_trade_time,volume_lots,gross_profit,gross_loss,net_pnl,avg_net_pnl,median_net_pnl,win_rate,avg_win,avg_loss,total_commission,total_swap,total_fee
0,EURUSD.a,21,2026-04-20 05:30:01+00:00,2026-04-24 20:00:07+00:00,26.25,1070.39,-563.29,507.10,24.15,65.70,66.67,76.46,-80.47,-91.98,0.0,0.0
1,US500.a,23,2026-04-21 15:38:31+00:00,2026-04-24 21:22:56+00:00,115.00,300.67,-346.64,-45.97,-2.00,9.78,52.17,25.06,-31.51,0.00,0.0,0.0
2,XAUUSD.a,48,2026-04-21 07:14:25+00:00,2026-04-24 18:34:56+00:00,4.80,1710.41,-2114.43,-404.02,-8.42,-1.32,50.00,71.27,-88.10,0.00,0.0,0.0


## OHLCV Candlestick Chart

Load 1-minute OHLCV data for the selected symbol and display it as an interactive candlestick chart with volume. Trade entries and exits from the MT5 report are overlaid as markers.


In [6]:
from pathlib import Path

_ohlcv_path = Path("../data") / f"{SYMBOL}_M1_520weeks.csv"

if not _ohlcv_path.exists():
    print(f"OHLCV file not found: {_ohlcv_path}")
    ohlcv = None
else:
    ohlcv = pd.read_csv(
        _ohlcv_path,
        parse_dates=["Time"],
        index_col="Time",
    )
    ohlcv.index = pd.to_datetime(ohlcv.index, utc=True)
    _start = pd.Timestamp(START_DATE, tz="UTC")
    _end   = pd.Timestamp(END_DATE,   tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    ohlcv  = ohlcv.loc[_start:_end]
    n_bars = len(ohlcv)
    print(f"Loaded {n_bars:,} 1-minute bars for {SYMBOL} ({ohlcv.index.min()} -> {ohlcv.index.max()})")


Loaded 6,886 1-minute bars for XAUUSD (2026-04-19 22:01:00+00:00 -> 2026-04-24 20:54:00+00:00)


In [7]:
if ohlcv is None or ohlcv.empty:
    print("No OHLCV data available to chart.")
else:
    # --- Pair IN/OUT deals for the selected symbol ---
    _sym_trades = (
        trades[trades["symbol"].str.startswith(SYMBOL)].copy()
        if not trades.empty
        else pd.DataFrame()
    )

    _entries = (
        _sym_trades[_sym_trades["entry_type"] == "IN"][
            ["position_id", "time", "price", "side"]
        ].rename(columns={"time": "entry_time", "price": "entry_price"})
    )
    _exits = (
        _sym_trades[_sym_trades["entry_type"].isin(["OUT", "OUT_BY", "INOUT"])][
            ["position_id", "time", "price", "net_pnl"]
        ].rename(columns={"time": "exit_time", "price": "exit_price"})
    )

    _pairs = _entries.merge(_exits, on="position_id", how="inner").copy()
    _pairs["result"] = _pairs["net_pnl"].apply(
        lambda x: "win" if x > 0 else ("loss" if x < 0 else "breakeven")
    )

    # Floor sub-minute timestamps so markers align with 1-min OHLCV bar open times.
    # MT5 deals carry second/millisecond precision; candlestick bars sit at exact
    # minute boundaries — without this floor, markers appear between candles.
    _pairs["entry_bar"] = _pairs["entry_time"].dt.floor("min")
    _pairs["exit_bar"]  = _pairs["exit_time"].dt.floor("min")

    _result_color = {"win": "lime", "loss": "red", "breakeven": "grey"}
    _side_color   = {"buy": "#00cc44", "sell": "#ff3333"}
    _entry_symbol = {"buy": "triangle-up", "sell": "triangle-down"}

    # --- Build chart ---
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.75, 0.25],
        vertical_spacing=0.04,
        subplot_titles=(f"{SYMBOL} — Candlestick", "Volume"),
    )

    fig.add_trace(
        go.Candlestick(
            x=ohlcv.index,
            open=ohlcv["Open"],
            high=ohlcv["High"],
            low=ohlcv["Low"],
            close=ohlcv["Close"],
            name=SYMBOL,
            increasing_line_color="#26a69a",
            decreasing_line_color="#ef5350",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=ohlcv.index,
            y=ohlcv["Volume"],
            name="Volume",
            marker_color="rgba(100,100,200,0.4)",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    if not _pairs.empty:
        # Connector lines: use floored bar times for x so lines track candles
        for _, row_data in _pairs.iterrows():
            col_ = _result_color.get(row_data["result"], "grey")
            fig.add_trace(
                go.Scatter(
                    x=[row_data["entry_bar"], row_data["exit_bar"]],
                    y=[row_data["entry_price"], row_data["exit_price"]],
                    mode="lines",
                    line=dict(color=col_, width=1, dash="dot"),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=1,
                col=1,
            )

        # Entry markers — x uses floored bar time, y uses actual fill price
        for side in ("buy", "sell"):
            _s = _pairs[_pairs["side"] == side]
            if _s.empty:
                continue
            fig.add_trace(
                go.Scatter(
                    x=_s["entry_bar"],
                    y=_s["entry_price"],
                    mode="markers",
                    name=f"Entry ({side})",
                    marker=dict(
                        symbol=_entry_symbol[side],
                        color=_side_color[side],
                        size=10,
                        line=dict(color="white", width=1),
                    ),
                    customdata=_s[["position_id", "entry_price", "entry_time"]].values,
                    hovertemplate=(
                        "Entry %{customdata[0]}<br>"
                        "Price: %{customdata[1]:.5f}<br>"
                        "Time: %{customdata[2]}<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )

        # Exit markers — x uses floored bar time, y uses actual fill price
        for result in ("win", "loss", "breakeven"):
            _r = _pairs[_pairs["result"] == result]
            if _r.empty:
                continue
            fig.add_trace(
                go.Scatter(
                    x=_r["exit_bar"],
                    y=_r["exit_price"],
                    mode="markers",
                    name=f"Exit ({result})",
                    marker=dict(
                        symbol="circle",
                        color=_result_color[result],
                        size=8,
                        line=dict(color="white", width=1),
                    ),
                    customdata=_r[["position_id", "exit_price", "net_pnl", "exit_time"]].values,
                    hovertemplate=(
                        "Exit %{customdata[0]}<br>"
                        "Price: %{customdata[1]:.5f}<br>"
                        "Net PnL: %{customdata[2]:.2f}<br>"
                        "Time: %{customdata[3]}<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )

    fig.update_layout(
        title=f"{SYMBOL} — OHLCV with Trades ({START_DATE} to {END_DATE})",
        template="plotly_white",
        height=750,
        hovermode="x unified",
        xaxis_rangeslider_visible=False,
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    fig.update_xaxes(title_text="Time (UTC)", row=2, col=1)
    fig.show()
    print(f"Trades overlaid: {len(_pairs)} matched entry/exit pairs")


Trades overlaid: 48 matched entry/exit pairs


## Daily net PnL and cumulative PnL

In [8]:
if closed_trades.empty:
    print("No closed trades available to chart.")
else:
    daily = (
        closed_trades.groupby(["trade_date", "symbol"], as_index=False)["net_pnl"]
        .sum()
        .sort_values(["trade_date", "symbol"])
    )
    daily_pivot = daily.pivot(index="trade_date", columns="symbol", values="net_pnl").fillna(0.0)
    cumulative = daily_pivot.cumsum()

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.10,
        subplot_titles=("Daily net PnL by symbol", "Cumulative net PnL by symbol"),
    )

    for symbol in daily_pivot.columns:
        fig.add_trace(
            go.Scatter(
                x=daily_pivot.index,
                y=daily_pivot[symbol],
                mode="lines+markers",
                name=f"{symbol} daily",
                legendgroup=str(symbol),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=cumulative.index,
                y=cumulative[symbol],
                mode="lines",
                name=f"{symbol} cumulative",
                legendgroup=str(symbol),
                showlegend=False,
            ),
            row=2,
            col=1,
        )

    zero_line_style = dict(color="black", width=1, dash="dash")
    fig.add_hline(y=0, line=zero_line_style, row=1, col=1)
    fig.add_hline(y=0, line=zero_line_style, row=2, col=1)
    fig.update_yaxes(title_text="Net PnL", row=1, col=1)
    fig.update_yaxes(title_text="Cumulative net PnL", row=2, col=1)
    fig.update_xaxes(title_text="Trade date", row=2, col=1)
    fig.update_layout(height=850, hovermode="x unified", template="plotly_white")
    fig.show()

    display(daily.tail(20))

,trade_date,symbol,net_pnl
0,2026-04-20,EURUSD.a,-17.55
1,2026-04-21,EURUSD.a,358.55
2,2026-04-21,US500.a,-9.67
3,2026-04-21,XAUUSD.a,-34.65
4,2026-04-22,EURUSD.a,-227.88
5,2026-04-22,US500.a,-67.67
6,2026-04-22,XAUUSD.a,310.55
7,2026-04-23,EURUSD.a,185.11
8,2026-04-23,US500.a,35.56
9,2026-04-23,XAUUSD.a,-183.38


In [9]:
if closed_trades.empty:
    print("No closed trades available to chart.")
else:
    plot_df = closed_trades.copy()
    plot_df["result"] = plot_df["net_pnl"].apply(lambda x: "win" if x > 0 else ("loss" if x < 0 else "breakeven"))
    plot_df["abs_profit"] = plot_df["net_pnl"].abs()

    # Remove top 5% outliers per symbol
    # p95_by_symbol = plot_df.groupby("symbol")["abs_profit"].transform(lambda s: s.quantile(0.95))
    # plot_df = plot_df[plot_df["abs_profit"] < p95_by_symbol]

    symbols = sorted(plot_df["symbol"].dropna().unique().tolist())
    if not symbols:
        print("No data left after filtering.")
    else:
        n_cols = 2
        n_rows = (len(symbols) + n_cols - 1) // n_cols

        fig = make_subplots(
            rows=n_rows,
            cols=n_cols,
            subplot_titles=[f"{s}" for s in symbols],
            vertical_spacing=0.10,
            horizontal_spacing=0.08,
        )

        for i, sym in enumerate(symbols):
            r = i // n_cols + 1
            c = i % n_cols + 1

            symbol_df = plot_df[plot_df["symbol"] == sym]
            max_val = symbol_df["abs_profit"].max()
            bin_size = max_val / 30 if max_val > 0 else 1

            fig.add_trace(
                go.Histogram(
                    x=symbol_df[symbol_df["result"] == "win"]["abs_profit"],
                    name="Wins",
                    marker_color="green",
                    opacity=0.75,
                    xbins=dict(start=0, end=max_val, size=bin_size),
                    legendgroup="wins",
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )
            fig.add_trace(
                go.Histogram(
                    x=symbol_df[symbol_df["result"] == "loss"]["abs_profit"],
                    name="Losses",
                    marker_color="red",
                    opacity=0.75,
                    xbins=dict(start=0, end=max_val, size=bin_size),
                    legendgroup="losses",
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )

            fig.update_xaxes(title_text="Absolute Profit", row=r, col=c)
            fig.update_yaxes(title_text="Count", row=r, col=c)

        fig.update_layout(
            title="Distribution of absolute profit for closed trades by symbol",
            barmode="overlay",
            template="plotly_white",
            height=max(400, 320 * n_rows),
        )
        fig.show()

## Best and worst closed trades

In [10]:
if closed_trades.empty:
    print("No closed trades available for ranking.")
else:
    cols = ["time", "symbol", "side", "entry_type", "volume", "price", "profit", "commission", "swap", "fee", "net_pnl", "comment"]
    print("Top 10 winners")
    display(closed_trades.sort_values("net_pnl", ascending=False)[cols].head(10))
    print("Top 10 losers")
    display(closed_trades.sort_values("net_pnl", ascending=True)[cols].head(10))

Top 10 winners


,time,symbol,side,entry_type,volume,price,profit,commission,swap,fee,net_pnl,comment
125,2026-04-23 16:40:52+00:00,XAUUSD.a,sell,OUT,0.10,4735.10000,102.13,0.00,0.0,0.0,102.13,[tp 4735.10]
123,2026-04-23 15:06:56+00:00,EURUSD.a,sell,OUT,1.25,1.16953,104.85,-4.38,0.0,0.0,100.47,[tp 1.16953]
27,2026-04-21 18:26:30+00:00,EURUSD.a,buy,OUT,1.25,1.17514,102.93,-4.38,0.0,0.0,98.55,[tp 1.17514]
155,2026-04-24 14:21:45+00:00,EURUSD.a,sell,OUT,1.25,1.17116,99.74,-4.38,0.0,0.0,95.36,[tp 1.17116]
39,2026-04-22 02:13:35+00:00,XAUUSD.a,sell,OUT,0.10,4727.05000,91.02,0.00,0.0,0.0,91.02,[tp 4727.05]
77,2026-04-22 18:17:39+00:00,XAUUSD.a,buy,OUT,0.10,4731.14000,89.96,0.00,0.0,0.0,89.96,[tp 4731.14]
38,2026-04-22 02:12:53+00:00,XAUUSD.a,sell,OUT,0.10,4726.69000,87.53,0.00,0.0,0.0,87.53,[tp 4726.69]
129,2026-04-23 18:19:27+00:00,EURUSD.a,sell,OUT,1.25,1.17124,89.02,-4.38,0.0,0.0,84.64,[tp 1.17124]
143,2026-04-24 05:10:12+00:00,XAUUSD.a,buy,OUT,0.10,4692.60000,83.07,0.00,0.0,0.0,83.07,[tp 4692.60]
180,2026-04-24 19:52:00+00:00,EURUSD.a,sell,OUT,1.25,1.17197,87.40,-4.38,0.0,0.0,83.02,[tp 1.17197]


Top 10 losers


,time,symbol,side,entry_type,volume,price,profit,commission,swap,fee,net_pnl,comment
170,2026-04-24 15:45:47+00:00,XAUUSD.a,sell,OUT,0.10,4697.5400,-143.08,0.00,0.0,0.0,-143.08,[sl 4697.54]
169,2026-04-24 15:44:33+00:00,XAUUSD.a,sell,OUT,0.10,4698.7400,-136.77,0.00,0.0,0.0,-136.77,[sl 4698.74]
23,2026-04-21 17:32:42+00:00,XAUUSD.a,sell,OUT,0.10,4769.3800,-130.10,0.00,0.0,0.0,-130.10,[sl 4769.38]
168,2026-04-24 15:37:31+00:00,EURUSD.a,sell,OUT,1.25,1.1707,-112.09,-4.38,0.0,0.0,-116.47,[sl 1.17070]
171,2026-04-24 15:46:43+00:00,XAUUSD.a,sell,OUT,0.10,4697.0900,-106.49,0.00,0.0,0.0,-106.49,[sl 4697.09]
74,2026-04-22 17:04:53+00:00,XAUUSD.a,sell,OUT,0.10,4745.4700,-104.39,0.00,0.0,0.0,-104.39,[sl 4745.47]
177,2026-04-24 18:34:56+00:00,XAUUSD.a,sell,OUT,0.10,4718.4900,-101.99,0.00,0.0,0.0,-101.99,[sl 4718.49]
101,2026-04-23 04:13:09+00:00,XAUUSD.a,sell,OUT,0.10,4737.3700,-101.28,0.00,0.0,0.0,-101.28,[sl 4737.37]
106,2026-04-23 04:23:27+00:00,XAUUSD.a,sell,OUT,0.10,4728.3500,-100.90,0.00,0.0,0.0,-100.90,[sl 4728.35]
108,2026-04-23 04:29:44+00:00,XAUUSD.a,sell,OUT,0.10,4727.7500,-99.10,0.00,0.0,0.0,-99.10,[sl 4727.75]


## Position-level analysis

In [11]:
from Learn.report import (
    build_position_pairs,
    compute_trade_quality_metrics,
    equity_curve_and_drawdown,
    compute_rolling_metrics,
    compute_hourly_performance,
    compute_weekday_performance,
    compute_side_performance,
    compute_mae_mfe,
)

pairs = build_position_pairs(trades)
metrics = compute_trade_quality_metrics(pairs)

print(f"Completed positions : {metrics.get('trade_count', 0):,}")
print(f"Win rate            : {metrics.get('win_rate', 0):.1%}")
print(f"Profit factor       : {metrics.get('profit_factor', 0):.2f}")
print(f"Expectancy          : {metrics.get('expectancy', 0):.2f}")
print(f"Avg win             : {metrics.get('avg_win', 0):.2f}")
print(f"Avg loss            : {metrics.get('avg_loss', 0):.2f}")
print(f"Payoff ratio        : {metrics.get('payoff_ratio', 0):.2f}")
print(f"Total commission    : {metrics.get('total_commission', 0):.2f}")
print(f"Commission/trade    : {metrics.get('commission_per_trade', 0):.2f}")
print(f"Total net PnL       : {metrics.get('total_net_pnl', 0):.2f}")


Completed positions : 92
Win rate            : 54.3%
Profit factor       : 0.99
Expectancy          : -0.38
Avg win             : 60.40
Avg loss            : -72.74
Payoff ratio        : 0.83
Total commission    : -183.96
Commission/trade    : -2.00
Total net PnL       : -34.87


## Equity curve and drawdown

In [12]:
if pairs.empty:
    print("No position pairs available for equity curve.")
else:
    curve, dd_metrics = equity_curve_and_drawdown(pairs)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.65, 0.35],
        subplot_titles=("Equity Curve", "Drawdown"),
        vertical_spacing=0.06,
    )

    fig.add_trace(go.Scatter(
        x=curve["exit_time"], y=curve["equity"],
        mode="lines", name="Equity", line=dict(color="steelblue", width=2),
    ), row=1, col=1)

    # Horizontal zero line on equity chart
    fig.add_hline(y=0, line_dash="dot", line_color="grey", row=1, col=1)

    # Vertical line at max drawdown point
    min_dd_idx = curve["drawdown"].idxmin()
    fig.add_vline(
        x=curve.loc[min_dd_idx, "exit_time"],
        line_dash="dash", line_color="red", row=1, col=1,
    )

    fig.add_trace(go.Scatter(
        x=curve["exit_time"], y=curve["drawdown"],
        mode="lines", name="Drawdown", fill="tozeroy",
        line=dict(color="red", width=1),
        fillcolor="rgba(220,50,50,0.25)",
    ), row=2, col=1)

    fig.update_layout(
        title="Equity Curve & Drawdown",
        height=600, showlegend=False,
        xaxis2_title="Exit Time",
        yaxis_title="Cumulative PnL",
        yaxis2_title="Drawdown",
    )
    fig.show()

    print(f"Final equity          : {dd_metrics['final_equity']:.2f}")
    print(f"Max drawdown          : {dd_metrics['max_drawdown']:.2f}")
    print(f"Max drawdown %        : {dd_metrics['max_drawdown_pct']:.2f}%")
    print(f"Longest DD streak     : {dd_metrics['longest_drawdown_trades']} trades")
    print(f"Calmar ratio          : {dd_metrics['calmar_ratio']:.2f}")


Final equity          : -34.87
Max drawdown          : -913.82
Max drawdown %        : -196.11%
Longest DD streak     : 67 trades
Calmar ratio          : -0.04


## Buy vs sell signal performance

In [13]:
if pairs.empty:
    print("No position pairs available for side analysis.")
else:
    side_perf = compute_side_performance(pairs)
    display(side_perf)

    # Imbalanced buy/sell performance can indicate directional bias in the model;
    # e.g. consistently higher win-rate on buys in a bull market suggests the model
    # may need re-training or class-weight adjustment to improve sell-signal quality.

    sides = side_perf["side"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=sides, y=side_perf["trade_count"], name="Trade Count",
        marker_color="steelblue",
    ))
    fig.add_trace(go.Bar(
        x=sides, y=side_perf["win_rate"] * 100, name="Win Rate (%)",
        marker_color="mediumseagreen",
    ))
    fig.add_trace(go.Bar(
        x=sides, y=side_perf["avg_pnl"], name="Avg PnL",
        marker_color="darkorange",
    ))
    fig.update_layout(
        barmode="group",
        title="Buy vs Sell Performance",
        xaxis_title="Side",
        height=420,
    )
    fig.show()


,side,trade_count,win_rate,avg_pnl,total_pnl,avg_duration_mins
0,buy,48,0.520833,-6.595208,-316.57,25.624306
1,sell,44,0.568182,6.402273,281.70,16.690909


## Time-of-day performance (UTC)

In [14]:
if pairs.empty:
    print("No position pairs available for hourly analysis.")
else:
    hourly = compute_hourly_performance(pairs)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Trade Count by Hour", "Avg PnL by Hour"),
        vertical_spacing=0.1,
    )

    fig.add_trace(go.Bar(
        x=hourly["hour"], y=hourly["trade_count"],
        name="Trade Count", marker_color="steelblue",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=hourly["hour"], y=hourly["avg_pnl"],
        name="Avg PnL",
        marker_color=["green" if v >= 0 else "red" for v in hourly["avg_pnl"]],
    ), row=2, col=1)

    fig.update_layout(
        title="Time-of-Day Performance (UTC)",
        height=560, showlegend=False,
        xaxis2_title="Hour of Day (UTC)",
        yaxis_title="Trade Count",
        yaxis2_title="Avg PnL",
    )
    fig.show()


## Day-of-week performance

In [15]:
if pairs.empty:
    print("No position pairs available for weekday analysis.")
else:
    weekday = compute_weekday_performance(pairs)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Trade Count by Day", "Avg PnL by Day"),
        vertical_spacing=0.1,
    )

    fig.add_trace(go.Bar(
        x=weekday["weekday"], y=weekday["trade_count"],
        name="Trade Count", marker_color="steelblue",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=weekday["weekday"], y=weekday["avg_pnl"],
        name="Avg PnL",
        marker_color=["green" if v >= 0 else "red" for v in weekday["avg_pnl"]],
    ), row=2, col=1)

    fig.update_layout(
        title="Day-of-Week Performance",
        height=560, showlegend=False,
        xaxis2_title="Day of Week",
        yaxis_title="Trade Count",
        yaxis2_title="Avg PnL",
    )
    fig.show()


## Trade duration distribution

In [16]:
if pairs.empty:
    print("No position pairs available for duration analysis.")
else:
    import numpy as np
    p99 = pairs["duration_mins"].quantile(0.99)
    dur_df = pairs[pairs["duration_mins"] <= p99].copy()

    wins_dur = dur_df[dur_df["result"] == "win"]["duration_mins"]
    losses_dur = dur_df[dur_df["result"] == "loss"]["duration_mins"]

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=("Duration Histogram (win vs loss)", "Duration Box Plot"),
        vertical_spacing=0.12,
    )

    fig.add_trace(go.Histogram(
        x=wins_dur, name="Win", opacity=0.65,
        marker_color="green", nbinsx=40,
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=losses_dur, name="Loss", opacity=0.65,
        marker_color="red", nbinsx=40,
    ), row=1, col=1)

    fig.add_trace(go.Box(
        y=wins_dur, name="Win", marker_color="green", boxmean=True,
    ), row=2, col=1)
    fig.add_trace(go.Box(
        y=losses_dur, name="Loss", marker_color="red", boxmean=True,
    ), row=2, col=1)

    fig.update_layout(
        barmode="overlay",
        title="Trade Duration Distribution",
        height=620,
        xaxis_title="Duration (mins)",
        yaxis_title="Count",
        yaxis2_title="Duration (mins)",
    )
    fig.show()

    med_win = wins_dur.median() if not wins_dur.empty else float('nan')
    med_loss = losses_dur.median() if not losses_dur.empty else float('nan')
    print(f"Median duration  wins : {med_win:.1f} mins")
    print(f"Median duration losses: {med_loss:.1f} mins")


Median duration  wins : 19.3 mins
Median duration losses: 9.2 mins


## Rolling win rate (last N trades)

In [17]:
ROLLING_WINDOW = 20  # configurable

if pairs.empty or len(pairs) < 2:
    print("Not enough trades for rolling metrics.")
else:
    rolling = compute_rolling_metrics(pairs, window=ROLLING_WINDOW)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(
            f"Rolling Win Rate (window={ROLLING_WINDOW})",
            f"Rolling Avg PnL (window={ROLLING_WINDOW})",
        ),
        vertical_spacing=0.1,
    )

    fig.add_trace(go.Scatter(
        x=rolling["exit_time"], y=rolling["rolling_win_rate"] * 100,
        mode="lines", name="Win Rate %", line=dict(color="steelblue", width=2),
    ), row=1, col=1)
    fig.add_hline(y=50, line_dash="dash", line_color="grey", row=1, col=1)

    fig.add_trace(go.Scatter(
        x=rolling["exit_time"], y=rolling["rolling_avg_pnl"],
        mode="lines", name="Avg PnL", line=dict(color="darkorange", width=2),
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)

    fig.update_layout(
        title=f"Rolling Performance (N={ROLLING_WINDOW})",
        height=560, showlegend=False,
        xaxis2_title="Exit Time",
        yaxis_title="Win Rate (%)",
        yaxis2_title="Avg PnL",
    )
    fig.show()


## Commission drag analysis

In [18]:
if pairs.empty:
    print("No position pairs available for commission analysis.")
else:
    gross_pnl = pairs["gross_pnl"].sum()
    total_commission = pairs["commission"].sum() + pairs["swap"].sum() + pairs["fee"].sum()
    net_pnl = pairs["net_pnl"].sum()
    commission_pct = abs(total_commission / gross_pnl * 100) if gross_pnl != 0 else float('nan')
    commission_per_trade = total_commission / len(pairs)

    print(f"Gross PnL            : {gross_pnl:.2f}")
    print(f"Total commission drag: {total_commission:.2f}")
    print(f"Net PnL              : {net_pnl:.2f}")
    print(f"Commission % of gross: {commission_pct:.2f}%")
    print(f"Commission per trade : {commission_per_trade:.2f}")

    # Waterfall chart
    fig = go.Figure(go.Waterfall(
        name="PnL Waterfall",
        orientation="v",
        measure=["relative", "relative", "total"],
        x=["Gross PnL", "Commission Drag", "Net PnL"],
        y=[gross_pnl, total_commission, 0],
        connector={"line": {"color": "rgb(63,63,63)"}},
        increasing={"marker": {"color": "green"}},
        decreasing={"marker": {"color": "red"}},
        totals={"marker": {"color": "steelblue"}},
    ))
    fig.update_layout(title="Commission Drag Waterfall", height=420, showlegend=False)
    fig.show()

    # Per-symbol breakdown if multiple symbols present
    if pairs["symbol"].nunique() > 1:
        sym_comm = (
            pairs.groupby("symbol")
            .agg(total_drag=("commission", lambda s: s.sum() + pairs.loc[s.index, "swap"].sum() + pairs.loc[s.index, "fee"].sum()))
            .reset_index()
        )
        fig2 = go.Figure(go.Bar(
            x=sym_comm["symbol"], y=sym_comm["total_drag"],
            marker_color=["green" if v >= 0 else "red" for v in sym_comm["total_drag"]],
        ))
        fig2.update_layout(
            title="Commission Drag by Symbol",
            xaxis_title="Symbol", yaxis_title="Total Drag",
            height=380,
        )
        fig2.show()


Gross PnL            : 149.09
Total commission drag: -183.96
Net PnL              : -34.87
Commission % of gross: 123.39%
Commission per trade : -2.00


## Maximum adverse / favorable excursion (MAE / MFE)

In [19]:
import numpy as np

MAE_MFE_SYMBOL = SYMBOL  # uses top-level config

if ohlcv is None or ohlcv.empty:
    print("No OHLCV data available for MAE/MFE analysis.")
elif pairs.empty:
    print("No position pairs available for MAE/MFE analysis.")
else:
    mae_mfe = compute_mae_mfe(pairs, ohlcv, symbol=MAE_MFE_SYMBOL)

    if mae_mfe.empty:
        print("MAE/MFE computation returned no results (check symbol filter or OHLCV date range).")
    else:
        wins_mm = mae_mfe[mae_mfe["result"] == "win"]
        losses_mm = mae_mfe[mae_mfe["result"] == "loss"]

        color_map = {"win": "green", "loss": "red", "breakeven": "grey"}
        colors = [color_map.get(r, "grey") for r in mae_mfe["result"]]

        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=("MFE vs MAE Scatter", "MAE Distribution (wins vs losses)"),
            vertical_spacing=0.12,
        )

        fig.add_trace(go.Scatter(
            x=mae_mfe["mae_pts"], y=mae_mfe["mfe_pts"],
            mode="markers",
            marker=dict(color=colors, size=7, opacity=0.75, line=dict(width=0.5, color="white")),
            name="Trades",
            text=mae_mfe["result"],
        ), row=1, col=1)

        # Diagonal reference line: MFE == |MAE| (i.e. MFE = -MAE)
        min_mae = float(mae_mfe["mae_pts"].min())
        fig.add_trace(go.Scatter(
            x=[min_mae, 0], y=[-min_mae, 0],
            mode="lines", name="MFE=|MAE|",
            line=dict(color="grey", dash="dash", width=1),
        ), row=1, col=1)

        fig.add_trace(go.Histogram(
            x=wins_mm["mae_pts"], name="Win MAE",
            marker_color="green", opacity=0.65, nbinsx=40,
        ), row=2, col=1)
        fig.add_trace(go.Histogram(
            x=losses_mm["mae_pts"], name="Loss MAE",
            marker_color="red", opacity=0.65, nbinsx=40,
        ), row=2, col=1)

        fig.update_layout(
            barmode="overlay",
            title=f"MAE / MFE Analysis — {MAE_MFE_SYMBOL}",
            height=700,
            xaxis_title="MAE (price pts)",
            yaxis_title="MFE (price pts)",
            xaxis2_title="MAE (price pts)",
            yaxis2_title="Count",
        )
        fig.show()

        # Interpretation: trades where MAE is large relative to MFE suggest poor entry
        # timing or TP/SL miscalibration — the position moved heavily against before any
        # favourable move, implying entries are late or stops are too wide.

        avg_mae_win = wins_mm["mae_pts"].mean() if not wins_mm.empty else float('nan')
        avg_mae_loss = losses_mm["mae_pts"].mean() if not losses_mm.empty else float('nan')
        avg_mfe_win = wins_mm["mfe_pts"].mean() if not wins_mm.empty else float('nan')
        med_ratio = mae_mfe["mfe_to_mae_ratio"].median()

        print(f"Avg MAE  (wins)      : {avg_mae_win:.4f} pts")
        print(f"Avg MAE  (losses)    : {avg_mae_loss:.4f} pts")
        print(f"Avg MFE  (wins)      : {avg_mfe_win:.4f} pts")
        print(f"Median MFE/MAE ratio : {med_ratio:.2f}")


Avg MAE  (wins)      : 2.4367 pts
Avg MAE  (losses)    : -17.6786 pts
Avg MFE  (wins)      : 12.0200 pts
Median MFE/MAE ratio : 0.77
